# MC-OCR: OCR toàn bộ hóa đơn bằng VietOCR (chạy trên Colab GPU)

**Vì sao cần notebook này:** dataset MC-OCR trên Kaggle chỉ có text của 4 trường đã gán nhãn;
text của các dòng còn lại bị rỗng (~49%). Ta dùng sẵn **vị trí các ô text** (box) trong `kie_data`
và chạy **VietOCR** (model public) để đọc lại chữ trong *mọi* ô.

**Cách chạy**
1. `Runtime → Change runtime type → T4 GPU`
2. Ở thanh bên trái, mở 🔑 **Secrets**, thêm secret tên `KAGGLE_API_TOKEN` = token Kaggle của bạn (chuỗi bắt đầu bằng `KGAT_`), bật *Notebook access*.
3. `Runtime → Run all`. Cuối cùng file **`mcocr_ocr.jsonl`** sẽ được tải về máy →
   chép vào `NLP_FN/data/raw/mcocr/`.

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-03a83ecb-767a-10e8-d0b7-f2ee922d0f93)


In [2]:
# 1) Cài thư viện. VietOCR cài --no-deps để tránh kéo theo imgaug (lỗi với numpy 2).
!pip install -q -U kaggle
!pip install -q --no-deps vietocr
!pip install -q einops gdown prefetch_generator pyyaml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vietocr 0.3.13 requires imgaug==0.4.0, which is not installed.
vietocr 0.3.13 requires lmdb>=1.0.0, which is not installed.
vietocr 0.3.13 requires albumentations==1.4.2, but you have albumentations 2.0.8 which is incompatible.
vietocr 0.3.13 requires einops==0.2.0, but you have einops 0.8.2 which is incompatible.
vietocr 0.3.13 requires gdown==4.4.0, but you have gdown 5.2.2 which is incompatible.
vietocr 0.3.13 requires pillow==10.2.0, but you have pillow 11.3.0 which is incompatible.
vietocr 0.3.13 requires prefetch-generator==1.0.1, but you have prefetch-generator 1.0.3 which is incompatible.


In [3]:
# 2) Tải MC-OCR từ Kaggle và chỉ giải nén phần cần dùng
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN").strip()

!kaggle datasets download -d domixi1989/vietnamese-receipts-mc-ocr-2021 -p /content
!unzip -q -o /content/vietnamese-receipts-mc-ocr-2021.zip "kie_data/*" -d /content/mcocr
!rm /content/vietnamese-receipts-mc-ocr-2021.zip
!ls /content/mcocr/kie_data/kie_data | head; ls /content/mcocr/kie_data/kie_data/boxes_and_transcripts | wc -l

Dataset URL: https://www.kaggle.com/datasets/domixi1989/vietnamese-receipts-mc-ocr-2021
License(s): unknown
100% 2.12G/2.12G [02:03<00:00, 18.4MB/s]

boxes_and_transcripts
image_list.csv
images
1151


In [4]:
# 3) Nạp VietOCR (vgg_transformer – model public, pretrained cho tiếng Việt)
import torch
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

cfg = Cfg.load_config_from_name("vgg_transformer")
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
cfg["predictor"]["beamsearch"] = False
ocr = Predictor(cfg)
print("device:", cfg["device"])

Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:03<00:00, 173MB/s]
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:145: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:02, 6349.57it/s] 


device: cuda


In [6]:
# 4) Đọc box của từng ảnh, cắt ảnh theo box và nhận dạng chữ
import glob, json, os
from PIL import Image
from tqdm.auto import tqdm

BASE = "/content/mcocr/kie_data/kie_data"
PAD = 3
BATCH = 64

def read_boxes(tsv):
    boxes = []
    for line in open(tsv, encoding="utf-8"):
        p = line.rstrip("\n").split(",")
        if len(p) < 10:
            continue
        poly = list(map(float, p[1:9]))   # x1,y1,...,x4,y4 ; bỏ text/label cũ
        boxes.append(poly)
    return boxes

def recognize(crops):
    out = []
    for i in range(0, len(crops), BATCH):
        chunk = crops[i:i + BATCH]
        try:
            out += ocr.predict_batch(chunk)
        except Exception:
            out += [ocr.predict(c) for c in chunk]
    return out

tsvs = sorted(glob.glob(f"{BASE}/boxes_and_transcripts/*.tsv"))
results = []
for tsv in tqdm(tsvs):
    img_id = os.path.basename(tsv)[:-4]
    img_path = f"{BASE}/images/{img_id}.jpg"
    if not os.path.exists(img_path):
        continue
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    polys = read_boxes(tsv)
    crops, keep = [], []
    for poly in polys:
        xs, ys = poly[0::2], poly[1::2]
        x0, y0 = max(0, int(min(xs)) - PAD), max(0, int(min(ys)) - PAD)
        x1, y1 = min(W, int(max(xs)) + PAD), min(H, int(max(ys)) + PAD)
        if x1 - x0 < 4 or y1 - y0 < 4:
            continue
        crops.append(img.crop((x0, y0, x1, y1)))
        keep.append(poly)
    texts = recognize(crops) if crops else []
    results.append({
        "img_id": img_id, "width": W, "height": H,
        "boxes": [{"poly": p, "text": t} for p, t in zip(keep, texts)],
    })

print(len(results), "ảnh,", sum(len(r["boxes"]) for r in results), "box")

  0%|          | 0/1151 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/vietocr/tool/translate.py:115: RuntimeWarning: invalid value encountered in divide
  char_probs = np.sum(char_probs, axis=-1)/(char_probs>0).sum(-1)


1151 ảnh, 40868 box


In [7]:
# 5) Xem thử 1 hoá đơn (sắp theo dòng từ trên xuống)
r = results[0]
for b in sorted(r["boxes"], key=lambda b: (min(b["poly"][1::2]), min(b["poly"][0::2]))):
    print(b["text"])

UNI
QLO
WWW.UNIQLO.COM
UNIOLO
Vincom Pham Ngọc Thạch
2, Đượng Phạm Ngọc Thạch
Phỏ Đường Thị Trấn Bằng Đa
Thành phố Hà Nội
0243 2045 T02
149,000
x1
2000137060319
ẢO THÙN
199,000
2000137434172
x
ẢO THÙN
499,000
2000139080131
1
ÁO SƠ-MI
2000137589353
249,000
ẢO THÙN
Sản phẩm lẻ
199,000
-50,000
2000141864590
249,000
1
QUẦN DÀI
2000142115066
499,000 T
1
QUẦN SHORT
249,000
2000139501377
ĐẦM PONTE
x1
2000138510042
399,000
ĐẦM RAYON
[Chiết khấu
50,000
Tổng cộng
2,442,000
2,442,000
Số thẻ
- 4444244551
Mã phế duyệt
009761
Đã bạo gồm thuế:
222,000
2,220,000
10.03 % của
C/H Pos
1366934
01/08/2020 20:13
Bạn được phục vụ bởi Hà
662020080120105600030144


In [9]:
# 6) Lưu và tải về máy
with open("/content/mcocr_ocr.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
from google.colab import files
files.download("/content/mcocr_ocr.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>